# MR2a — Address Store MapReduce
**Input**: `manhattan_liquor_stores_geocoded.csv`
**Output**: `address_store_mr.csv`

Schema: `zone_id, store_count, active_licenses, outdated_licenses, avg_lat, avg_lon`

In [1]:
import h3
import pandas as pd
import numpy as np
from collections import defaultdict
from datetime import datetime

INPUT  = 'manhattan_liquor_stores_geocoded.csv'
OUTPUT = 'address_store_mr.csv'
H3_RES = 10

df = pd.read_csv(INPUT)
df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df = df.dropna(subset=['latitude','longitude'])

print(f'Input rows: {len(df):,}')

Input rows: 6,925


In [2]:
# ── MAP ───────────────────────────────────────────────────────────────────────
# Emit: key=zone_id, val=(license_status, lat, lon)

def get_license_status(exp_raw):
    try:
        exp = pd.to_datetime(exp_raw)
        return 'Active' if exp >= pd.Timestamp.today() else 'License Outdated'
    except:
        return 'Unknown'

def mapper(row):
    try:
        zone_id = h3.latlng_to_cell(float(row['latitude']), float(row['longitude']), H3_RES)
    except:
        return None, None
    val = {
        'status': get_license_status(row.get('license_expiration_date', '')),
        'lat':    float(row['latitude']),
        'lon':    float(row['longitude']),
    }
    return zone_id, val

mapped = [mapper(row) for _, row in df.iterrows()]
mapped = [(k,v) for k,v in mapped if k is not None]
print(f'Mapped pairs: {len(mapped):,}')

Mapped pairs: 6,925


In [ ]:
# ── REDUCE ────────────────────────────────────────────────────────────────────
# Per zone_id: count stores, active, outdated, avg lat/lon

acc = defaultdict(lambda: {'store_count':0,'active':0,'outdated':0,'lat':[],'lon':[]})

for zone_id, val in mapped:
    a = acc[zone_id]
    a['store_count'] += 1
    if val['status'] == 'Active':            a['active']   += 1
    elif val['status'] == 'License Outdated': a['outdated'] += 1
    a['lat'].append(val['lat'])
    a['lon'].append(val['lon'])

rows = []
for zone_id, a in acc.items():
    rows.append({
        'zone_id':          zone_id,
        'store_count':      a['store_count'],
        'active_licenses':  a['active'],
        'outdated_licenses':a['outdated'],
        'avg_lat':          float(np.mean(a['lat'])),
        'avg_lon':          float(np.mean(a['lon'])),
    })

# ── OUTPUT ────────────────────────────────────────────────────────────────────
out = pd.DataFrame(rows).sort_values('zone_id').reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

print(f'Output rows : {len(out):,}')
print(f'Active zones: {(out["active_licenses"]>0).sum():,}')
print(f'Saved → {OUTPUT}')
out.head()

Output rows : 1,823
Active zones: 270
Saved → address_store_mr.csv


,zone_id,store_count,active_licenses,outdated_licenses,avg_lat,avg_lon
0,8a2a1008804ffff,3,0,3,40.793831,-73.972384
1,8a2a10088067fff,4,0,4,40.795648,-73.971053
2,8a2a1008814ffff,6,1,5,40.796997,-73.970071
3,8a2a10088167fff,5,1,4,40.798422,-73.969041
4,8a2a1008816ffff,5,0,5,40.797746,-73.969523
